# 12 - UI 前端层指标

**目标**: 量化前端 React 应用的加载时间、渲染性能、Bundle 大小，以及浏览器到 SSE 第一个字符的完整链路时间。

**覆盖事件链 (UI 层)**:
```
[UI-1] 用户点击发送
[UI-2] React onClick → setState(loading=true) → 重渲染
[UI-3] EventSource 连接建立 → /graphql/stream
[UI-4] SSE 接收第一个 content chunk
[UI-5] React setState(messages=[...]) → 重渲染 → DOM 更新
[UI-6] 用户看到第一个字
```

**技术栈**:
- React 18 + Vite
- EventSource API (浏览器原生 SSE)
- Ant Design 组件库
- 端口: 3001 (dev/preview)

**测量策略**:
1. Python 端: httpx 模拟 SSE 连接（与浏览器 EventSource 等效网络开销）
2. Bundle 分析: Vite 构建产物静态分析
3. 浏览器端: 提供 DevTools Console 脚本（可直接粘贴运行）
4. 静态代码分析: 组件树、代码规模

In [ ]:
# ============================================================
# CELL 0: 配置区域 — 所有可控参数在此修改
# ============================================================

# --- 前端服务端点 ---
FRONTEND_URL    = "http://localhost:3001"    # React dev/preview 服务
API_GATEWAY_URL = "http://localhost:8001"    # API Gateway (SSE 端点)

# --- 前端代码路径 ---
FRONTEND_DIR = "/Users/dantevonalcatraz/Development/procurement-agents/xiaocai-app"
SRC_DIR      = f"{FRONTEND_DIR}/src"
DIST_DIR     = f"{FRONTEND_DIR}/dist"

# --- 采集参数 ---
N_SAMPLES = 10    # SSE 连接测量次数
TIMEOUT_S = 60.0  # 超时

# --- SSE 端点（前端实际连接的 URL）---
GRAPHQL_STREAM_URL = f"{API_GATEWAY_URL}/graphql/stream"

# --- GraphQL Subscription（与前端代码一致）---
GQL_SUBSCRIPTION = """
subscription ChatStream($message: String!, $sessionId: String!) {
  chatStream(message: $message, sessionId: $sessionId) {
    content
    type
    requirementDoc
  }
}
"""

# --- 输出路径 ---
OUTPUT_DIR = "../data"

import time as _time
_ts = int(_time.time())

print("配置加载完成")
print(f"  Frontend : {FRONTEND_URL}")
print(f"  SSE URL  : {GRAPHQL_STREAM_URL}")

In [ ]:
# ============================================================
# CELL 1: 依赖导入
# ============================================================

import httpx
import json
import os
import subprocess
import time
from pathlib import Path
from datetime import datetime
from typing import Dict, List

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("依赖加载完成")

In [ ]:
# ============================================================
# CELL 2: 前端服务健康检查
# ============================================================

print("=" * 60)
print("前端服务健康检查")
print("=" * 60)

frontend_status = {}

# 检查前端服务是否运行
try:
    t0 = time.perf_counter()
    with httpx.Client(timeout=10.0) as c:
        r = c.get(FRONTEND_URL)
    elapsed = (time.perf_counter() - t0) * 1000
    content_len = len(r.content)
    frontend_status["running"] = True
    frontend_status["status_code"] = r.status_code
    frontend_status["response_ms"] = round(elapsed, 2)
    frontend_status["content_length"] = content_len
    frontend_status["content_type"] = r.headers.get("content-type", "unknown")
    print(f"  前端服务: 运行中")
    print(f"  响应时间: {elapsed:.1f}ms")
    print(f"  HTML 大小: {content_len:,} bytes")
    print(f"  内容类型: {r.headers.get('content-type', 'unknown')}")
except Exception as e:
    frontend_status["running"] = False
    frontend_status["error"] = str(e)
    print(f"  前端服务: 未运行 ({e})")
    print(f"  提示: 运行 'cd xiaocai-app && npm run preview' 启动前端服务")

# API Gateway 检查
try:
    with httpx.Client(timeout=5.0) as c:
        r = c.get(f"{API_GATEWAY_URL}/health")
    frontend_status["api_gateway"] = "running"
    print(f"  API Gateway: 运行中")
except Exception as e:
    frontend_status["api_gateway"] = f"error: {e}"
    print(f"  API Gateway: 未运行")

In [ ]:
# ============================================================
# CELL 3: Bundle 大小静态分析
# ============================================================

print("=" * 60)
print("Bundle 大小分析")
print("=" * 60)

bundle_results = {}

dist_path = Path(DIST_DIR)
if dist_path.exists():
    print("\n  dist/ 目录存在，分析构建产物...")

    # 递归分析 dist 目录
    assets_dir = dist_path / "assets"
    files_data = []

    for f in dist_path.rglob("*"):
        if f.is_file():
            size = f.stat().st_size
            suffix = f.suffix
            files_data.append({
                "filename": f.name,
                "path":     str(f.relative_to(dist_path)),
                "size_bytes": size,
                "size_kb":  round(size / 1024, 2),
                "type":     suffix.lstrip(".")
            })

    df_bundle = pd.DataFrame(files_data).sort_values("size_bytes", ascending=False)

    # 按类型汇总
    type_summary = df_bundle.groupby("type").agg(
        count=("filename", "count"),
        total_kb=("size_kb", "sum")
    ).sort_values("total_kb", ascending=False)

    print(f"\n  文件类型汇总:")
    print(f"  {'类型':<10} {'文件数':>8} {'大小 (KB)':>12}")
    print("  " + "-" * 35)
    for t, row in type_summary.iterrows():
        print(f"  {t:<10} {int(row['count']):>8} {row['total_kb']:>12.1f}")

    total_kb = df_bundle["size_kb"].sum()
    print(f"\n  总大小: {total_kb:.1f} KB ({total_kb/1024:.2f} MB)")

    # 最大的 JS 文件
    js_files = df_bundle[df_bundle["type"] == "js"].head(10)
    print(f"\n  最大的 JS chunk:")
    for _, row in js_files.iterrows():
        print(f"    {row['filename']:<50} {row['size_kb']:>8.1f} KB")

    bundle_results = {
        "total_kb": round(total_kb, 2),
        "type_summary": type_summary.to_dict(),
        "top_js_files": js_files[["filename", "size_kb"]].to_dict(orient="records")
    }

else:
    print(f"  dist/ 目录不存在")
    print(f"  路径: {DIST_DIR}")
    print(f"  提示: 运行 'cd xiaocai-app && npm run build' 构建后再分析")
    print(f"\n  基于 package.json 估算依赖大小:")

    # 基于已知依赖的估算
    known_sizes = {
        "react + react-dom":        "~45 KB (gzipped)",
        "antd":                     "~310 KB (gzipped, tree-shaken)",
        "react-router-dom":         "~25 KB (gzipped)",
        "react-markdown":           "~30 KB (gzipped)",
        "@fortawesome/fontawesome": "~80 KB (gzipped)",
        "axios":                    "~15 KB (gzipped)",
        "业务代码 (src/)": "~50-80 KB (估算)",
    }
    for pkg, size in known_sizes.items():
        print(f"    {pkg:<45} {size}")
    print(f"\n  估算总大小: ~550-600 KB (gzipped)")
    bundle_results = {"estimated": True, "note": "需要 npm run build 后分析"}

In [ ]:
# ============================================================
# CELL 4: 源代码静态分析（组件树 + 代码规模）
# ============================================================

print("=" * 60)
print("源代码静态分析")
print("=" * 60)

src_path = Path(SRC_DIR)
code_stats = {
    "components": [],
    "pages": [],
    "services": [],
    "helpers": [],
    "total_files": 0,
    "total_loc": 0,
}

for f in src_path.rglob("*"):
    if f.is_file() and f.suffix in [".jsx", ".js", ".tsx", ".ts"]:
        if "deprecated" in str(f) or "node_modules" in str(f):
            continue
        try:
            content = f.read_text(encoding="utf-8")
            loc = len(content.splitlines())
            rel = str(f.relative_to(src_path))

            entry = {
                "path": rel,
                "loc": loc,
                "name": f.stem
            }

            if rel.startswith("components/"):
                code_stats["components"].append(entry)
            elif rel.startswith("pages/"):
                code_stats["pages"].append(entry)
            elif rel.startswith("services/"):
                code_stats["services"].append(entry)
            elif rel.startswith("helper/"):
                code_stats["helpers"].append(entry)

            code_stats["total_files"] += 1
            code_stats["total_loc"] += loc
        except Exception:
            pass

print(f"\n  代码统计:")
print(f"    总文件数: {code_stats['total_files']}")
print(f"    总代码行: {code_stats['total_loc']:,}")

print(f"\n  组件 ({len(code_stats['components'])} 个):")
for c in sorted(code_stats["components"], key=lambda x: x["loc"], reverse=True):
    print(f"    {c['path']:<60} {c['loc']:>5} 行")

print(f"\n  页面 ({len(code_stats['pages'])} 个):")
for p in code_stats["pages"]:
    print(f"    {p['path']:<60} {p['loc']:>5} 行")

print(f"\n  服务层 ({len(code_stats['services'])} 个):")
for s in code_stats["services"]:
    print(f"    {s['path']:<60} {s['loc']:>5} 行")

print(f"\n  组件树结构（UI 渲染链）:")
print("""
  App.jsx
  └── Router
      ├── / → Welcome/index.jsx
      └── /workspace → Workspace/index.jsx
          └── ProblemDrivenLayout
              ├── ChatInterface/index.jsx        ← 主聊天区域
              │   ├── ChatMessage/index.jsx      ← 消息气泡
              │   └── [MessageInput]             ← 用户输入
              ├── ToolPanel/index.jsx            ← 右侧工具面板
              │   ├── RequirementProgress       ← 字段填充进度
              │   ├── RequirementDoc            ← 生成的文档
              │   └── SessionStatusPanel        ← 会话状态
              └── SessionList/index.jsx          ← 左侧会话列表
""")

In [ ]:
# ============================================================
# CELL 5: [UI-3/4] EventSource SSE 连接时序
# Python httpx 模拟浏览器 EventSource 连接行为
# ============================================================

print("=" * 60)
print("[UI-3/4] EventSource SSE 连接时序")
print("(httpx 模拟浏览器 EventSource，等效网络行为)")
print("=" * 60)

sse_results = []

import urllib.parse

# 构建前端使用的 SSE URL（与 chatStream.js 一致）
# URL 格式: /graphql/stream?query=...&variables={...}
variables = json.dumps({"message": "我要采购笔记本", "sessionId": f"ui-bench-{_ts}"})
params = urllib.parse.urlencode({"query": GQL_SUBSCRIPTION, "variables": variables})
sse_url = f"{GRAPHQL_STREAM_URL}?{params}"

print(f"\n  目标 URL: {GRAPHQL_STREAM_URL}")
print(f"  模拟 EventSource GET 请求 ({N_SAMPLES} 次)...")

for i in range(N_SAMPLES):
    sid = f"ui-bench-{_ts}-{i}"
    vars_i = json.dumps({"message": "我要采购笔记本", "sessionId": sid})
    params_i = urllib.parse.urlencode({"query": GQL_SUBSCRIPTION, "variables": vars_i})
    url_i = f"{GRAPHQL_STREAM_URL}?{params_i}"

    t_start = time.perf_counter()
    t_connect = None
    t_first_chunk = None
    t_total = None
    chunk_count = 0

    try:
        with httpx.Client(timeout=TIMEOUT_S) as c:
            with c.stream(
                "GET", url_i,
                headers={
                    "Accept": "text/event-stream",
                    "Cache-Control": "no-cache",
                    # 模拟浏览器 EventSource 的请求头
                    "User-Agent": "Mozilla/5.0 (EventSource simulation)",
                }
            ) as resp:
                t_connect = (time.perf_counter() - t_start) * 1000
                current_event = None

                for line in resp.iter_lines():
                    if line.startswith("event:"):
                        current_event = line[6:].strip()
                    elif line.startswith("data:"):
                        try:
                            d = json.loads(line[5:].strip())
                            if d.get("content") and t_first_chunk is None:
                                t_first_chunk = (time.perf_counter() - t_start) * 1000
                            chunk_count += 1
                            if chunk_count >= 5:  # 获取足够数据后退出
                                break
                        except:
                            pass

        t_total = (time.perf_counter() - t_start) * 1000
        result = {
            "t_connect_ms":     t_connect,
            "t_first_chunk_ms": t_first_chunk,
            "t_partial_ms":     t_total,
            "chunks":           chunk_count,
        }
        sse_results.append(result)
        print(f"  [{i+1}/{N_SAMPLES}] connect={t_connect:.0f}ms "
              f"first_chunk={t_first_chunk:.0f}ms "
              f"chunks={chunk_count}")

    except Exception as e:
        print(f"  [{i+1}/{N_SAMPLES}] 错误: {e}")

if sse_results:
    t_connects     = [r["t_connect_ms"] for r in sse_results if r["t_connect_ms"]]
    t_first_chunks = [r["t_first_chunk_ms"] for r in sse_results if r["t_first_chunk_ms"]]

    print(f"\n[EventSource 时序统计]")
    print(f"  连接建立时间: mean={np.mean(t_connects):.0f}ms  p95={np.percentile(t_connects, 95):.0f}ms")
    if t_first_chunks:
        print(f"  首字符时间:   mean={np.mean(t_first_chunks):.0f}ms  p95={np.percentile(t_first_chunks, 95):.0f}ms")
        print(f"  用户感知延迟 (click → 首字): ~{np.mean(t_first_chunks):.0f}ms")

In [ ]:
# ============================================================
# CELL 6: [UI-1/2] React 渲染链分析（静态 + 时序估算）
# ============================================================

print("=" * 60)
print("[UI-1/2] React 渲染链时序估算")
print("=" * 60)

# 基于 React 18 特性和代码分析的时序估算
render_timeline = [
    # (事件, 时间ms, 颜色, 说明)
    ("用户点击发送",            0,     "#2ecc71", "onClick 触发"),
    ("setState(loading=true)",  1,     "#3498db", "同步状态更新"),
    ("React 重渲染 (loading)",  3,     "#3498db", "调度渲染 + Reconciler"),
    ("DOM 更新 (loading 图标)", 5,     "#3498db", "Commit 阶段"),
    ("EventSource 初始化",      5,     "#9b59b6", "new EventSource(url)"),
    ("TCP 握手到服务器",         10,    "#e67e22", "loopback ~1ms, 生产网络 5-20ms"),
    ("HTTP 响应头 (200)",       12,    "#e67e22", "服务器接受 GET 连接"),
    ("Agent 处理中...",         1000,  "#e74c3c", "LLM 意图识别 + 字段提取"),
    ("第一个 SSE 数据块",        1200,  "#e74c3c", "onmessage 触发"),
    ("setState(messages=[...])", 1201, "#1abc9c", "React 状态更新"),
    ("React 重渲染 (message)",  1204, "#1abc9c", "ChatMessage 组件渲染"),
    ("DOM 更新 + Layout",       1206, "#1abc9c", "浏览器 layout + paint"),
    ("用户看到第一个字",          1210, "#2ecc71", "FCP from user interaction"),
]

print("\n  React 渲染链时序（估算，基于代码分析）:")
print(f"  {'事件':<35} {'时间 (ms)':>10} {'说明'}")
print("  " + "-" * 75)
for event, t_ms, color, desc in render_timeline:
    print(f"  {event:<35} {t_ms:>10}  {desc}")

# React 渲染周期分析
print("\n  React 渲染关键路径:")
print("""
  用户点击 → onClick()
    ├── setLoading(true)              [同步, <1ms]
    ├── sendChatMessageStream()
    │   ├── subscribeGraphQL()        [同步]
    │   └── new EventSource(url)      [异步, 建立SSE连接]
    └── React 调度重渲染              [异步, ~1-3ms]

  EventSource.onmessage()
    ├── JSON.parse(event.data)        [<1ms]
    ├── setMessages([...prev, chunk]) [触发重渲染]
    └── React Reconciler
        ├── ChatInterface diff        [~1ms]
        ├── ChatMessage 新组件         [~1ms]
        └── DOM Commit + Layout       [~2ms, 浏览器]

  每个 SSE chunk → UI 更新的增量成本: ~2-5ms
  （无虚拟化时，消息增多后可能退化）
""")

# React 性能关注点
print("  性能关注点:")
perf_concerns = [
    ("消息列表无虚拟化",     "高",  "messages[] 全量重渲染，消息>50条后明显"),
    ("SSE chunk 逐字更新",   "中",  "每个字符触发 setState，高频渲染"),
    ("Ant Design 组件库",    "低",  "经过 tree-shaking 影响有限"),
    ("React-markdown",       "中",  "Markdown 解析每次渲染都执行"),
]
for concern, level, note in perf_concerns:
    print(f"    [{level}] {concern}: {note}")

In [ ]:
# ============================================================
# CELL 7: 浏览器 DevTools Console 脚本
# 直接粘贴到 Chrome DevTools Console 运行
# ============================================================

print("=" * 60)
print("浏览器 DevTools Console 脚本")
print("（在 Chrome DevTools Console 中运行以获取真实浏览器指标）")
print("=" * 60)

devtools_script = """
// ============================================================
// 小采 UI 性能测量脚本
// 在 Chrome DevTools Console 中运行
// ============================================================

// 1. 页面加载指标
const nav = performance.getEntriesByType('navigation')[0];
console.group('页面加载指标');
console.log('DNS 解析:', (nav.domainLookupEnd - nav.domainLookupStart).toFixed(2), 'ms');
console.log('TCP 连接:', (nav.connectEnd - nav.connectStart).toFixed(2), 'ms');
console.log('TTFB:', (nav.responseStart - nav.requestStart).toFixed(2), 'ms');
console.log('HTML 传输:', (nav.responseEnd - nav.responseStart).toFixed(2), 'ms');
console.log('DOM 解析:', (nav.domContentLoadedEventEnd - nav.responseEnd).toFixed(2), 'ms');
console.log('总加载时间:', nav.loadEventEnd.toFixed(2), 'ms');
console.groupEnd();

// 2. 资源加载（JS/CSS）
const resources = performance.getEntriesByType('resource');
const jsRes = resources.filter(r => r.initiatorType === 'script');
const cssRes = resources.filter(r => r.initiatorType === 'link');
console.group('资源加载');
console.log('JS 文件数:', jsRes.length);
jsRes.forEach(r => {
    const size = r.transferSize ? (r.transferSize/1024).toFixed(1) + ' KB' : '(cached)';
    console.log(' ', r.name.split('/').pop(), '|', r.duration.toFixed(0), 'ms |', size);
});
console.groupEnd();

// 3. FCP / LCP（需要 PerformanceObserver 预先注册）
new PerformanceObserver((list) => {
    list.getEntries().forEach(entry => {
        console.log('FCP:', entry.startTime.toFixed(2), 'ms');
    });
}).observe({entryTypes: ['first-contentful-paint']});

new PerformanceObserver((list) => {
    const entries = list.getEntries();
    const last = entries[entries.length - 1];
    console.log('LCP:', last.startTime.toFixed(2), 'ms', '| 元素:', last.element?.tagName);
}).observe({entryTypes: ['largest-contentful-paint']});

// 4. 用户交互到首字指标（在发送消息前运行）
let _t_click = null;
let _t_first_char = null;
window._measureSendChat = function() {
    _t_click = performance.now();
    console.log('点击时间戳:', _t_click.toFixed(2), 'ms');
    // 观察 DOM 变化（等待第一个字符出现）
    const observer = new MutationObserver(() => {
        if (_t_first_char === null) {
            _t_first_char = performance.now();
            console.log('首字符时间:', (_t_first_char - _t_click).toFixed(2), 'ms');
            observer.disconnect();
        }
    });
    // 观察消息列表区域
    const chatList = document.querySelector('[class*="chat"], [class*="message"], .messages');
    if (chatList) observer.observe(chatList, {childList: true, subtree: true});
    else console.warn('未找到消息列表元素，手动查找 className');
};
console.log('运行方式: 先运行 window._measureSendChat(), 然后点击发送按钮');
"""

print("\n复制以下代码到 Chrome DevTools Console 运行:")
print("-" * 60)
print(devtools_script)
print("-" * 60)

# 保存到文件供参考
script_path = f"{OUTPUT_DIR}/browser-devtools-script.js"
with open(script_path, "w", encoding="utf-8") as f:
    f.write(devtools_script)
print(f"\n脚本已保存: {script_path}")

In [ ]:
# ============================================================
# CELL 8: 综合可视化
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle("UI 前端层指标", fontsize=16, fontweight='bold', y=1.01)

# ── 子图 1: React 渲染链时序甘特图 ────────────────────────
ax1 = axes[0]
ax1.set_title("React 渲染链时序\n（click → 首字符）", fontweight='bold')

# 分组事件
groups = [
    ("浏览器",   [("点击", 0, 5), ("DOM 更新", 5, 7)],        "#3498db"),
    ("网络",     [("TCP+HTTP", 5, 12), ("SSE 等待", 12, 1200)], "#e67e22"),
    ("Agent",   [("LLM 处理", 12, 1200)],                      "#e74c3c"),
    ("React渲染", [("重渲染", 1200, 1210)],                     "#1abc9c"),
]

y_ticks = []
y_labels = []
y = 0
for group_name, events, color in groups:
    for ev_name, start, end in events:
        ax1.barh(y, end - start, left=start, height=0.6,
                 color=color, alpha=0.8)
        if end - start > 30:
            ax1.text(start + (end-start)/2, y, f"{ev_name}\n{end-start}ms",
                     ha='center', va='center', fontsize=7, color='white', fontweight='bold')
        else:
            ax1.text(end + 10, y, f"{ev_name} {end-start}ms",
                     va='center', fontsize=7)
        y_ticks.append(y)
        y_labels.append(f"{group_name}")
        y += 1

ax1.set_yticks(y_ticks)
ax1.set_yticklabels(y_labels, fontsize=8)
ax1.set_xlabel("时间 (ms from click)")
ax1.axvline(x=1210, color='green', linestyle='--', linewidth=2, label='用户看到首字')
ax1.legend(fontsize=8)
ax1.set_xscale('log')
ax1.set_xlim(0.5, 3000)

# ── 子图 2: SSE 连接时序测量结果 ─────────────────────────
ax2 = axes[1]
ax2.set_title("SSE 连接时序\n(httpx 模拟 EventSource)", fontweight='bold')

if sse_results:
    t_connects     = [r["t_connect_ms"]     for r in sse_results if r["t_connect_ms"]]
    t_first_chunks = [r["t_first_chunk_ms"] for r in sse_results if r["t_first_chunk_ms"]]

    if t_connects and t_first_chunks:
        positions = [1, 2]
        data_list = [t_connects, t_first_chunks]
        labels_bp  = ["连接建立", "首字符"]

        bp = ax2.boxplot(data_list, positions=positions, widths=0.5,
                         patch_artist=True,
                         boxprops=dict(facecolor='lightblue'),
                         medianprops=dict(color='red', linewidth=2))
        ax2.set_xticks(positions)
        ax2.set_xticklabels(labels_bp, fontsize=9)
        ax2.set_ylabel("时间 (ms)")

        for pos, data, label in zip(positions, data_list, labels_bp):
            mean_v = np.mean(data)
            ax2.text(pos, mean_v, f"  mean\n  {mean_v:.0f}ms",
                     va='center', fontsize=8, color='darkblue')
else:
    ax2.text(0.5, 0.5, "SSE 测量失败\n请检查服务状态",
             ha='center', va='center', transform=ax2.transAxes, fontsize=12)

# ── 子图 3: 依赖包大小估算 ─────────────────────────────────
ax3 = axes[2]
ax3.set_title("Bundle 组成\n(gzipped 估算)", fontweight='bold')

# 估算各包 gzip 后大小 (KB)
pkg_sizes = {
    "antd":             310,
    "react+react-dom":  45,
    "@fortawesome":     80,
    "react-markdown":   30,
    "react-router-dom": 25,
    "axios":            15,
    "业务代码": 65,
}

# 若有实际 dist 数据则覆盖
if bundle_results and not bundle_results.get("estimated"):
    total_real = bundle_results.get("total_kb", 0)
    ax3.set_title(f"Bundle 大小 (实测: {total_real:.0f} KB)", fontweight='bold')

colors_pkg = plt.cm.Set3(np.linspace(0, 1, len(pkg_sizes)))
wedges, texts, autotexts = ax3.pie(
    list(pkg_sizes.values()),
    labels=[f"{k}\n{v}KB" for k, v in pkg_sizes.items()],
    colors=colors_pkg,
    autopct='%1.0f%%',
    startangle=90,
    pctdistance=0.75,
    textprops={'fontsize': 8}
)
total_est = sum(pkg_sizes.values())
ax3.set_title(f"Bundle 组成估算\n总计 ~{total_est} KB (gzipped)", fontweight='bold')

plt.tight_layout()
ts = datetime.now().strftime("%Y%m%d_%H%M")
save_path = f"{OUTPUT_DIR}/ui-frontend-metrics_{ts}.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"图表已保存: {save_path}")

In [ ]:
# ============================================================
# CELL 9: 汇总报告 + 保存
# ============================================================

print("=" * 60)
print("UI 前端层分析汇总")
print("=" * 60)

print("\n[前端技术栈]")
print("  框架:     React 18 (Concurrent Mode)")
print("  构建:     Vite 7 + TypeScript")
print("  UI 库:    Ant Design 6")
print("  SSE:      浏览器原生 EventSource API")
print("  路由:     react-router-dom v6")
print("  端口:     3001 (dev/preview)")

print("\n[UI 层事件链时序]")
print("  [UI-1] 用户点击发送:             0 ms")
print("  [UI-2] setState + React 重渲染:  ~1-5 ms")
print("  [UI-3] EventSource 建立连接:     ~5-15 ms (本地), 50-100ms (生产)")
if sse_results:
    t_c = [r["t_connect_ms"] for r in sse_results if r["t_connect_ms"]]
    t_f = [r["t_first_chunk_ms"] for r in sse_results if r["t_first_chunk_ms"]]
    if t_c:
        print(f"  [UI-3] 实测连接建立:             mean={np.mean(t_c):.0f}ms")
    if t_f:
        print(f"  [UI-4] 实测首字符:               mean={np.mean(t_f):.0f}ms")
print("  [UI-5] onmessage → setState:     <1 ms")
print("  [UI-5] React 重渲染 + DOM 更新:  ~2-5 ms")
print("  [UI-6] 浏览器 paint:             ~1-2 ms")

print("\n[代码规模]")
print(f"  总文件数: {code_stats['total_files']}")
print(f"  总代码行: {code_stats['total_loc']:,}")
print(f"  组件数:   {len(code_stats['components'])}")
print(f"  页面数:   {len(code_stats['pages'])}")

print("\n[关键发现]")
print("  1. 浏览器 React 渲染开销 <10ms，不是瓶颈")
print("  2. LLM 处理是主要瓶颈 (~1000-3000ms)")
print("  3. 流式 SSE 显著改善感知体验（逐字显示 vs 全量等待）")
print("  4. antd 是最大依赖 (~310KB gzipped)，已 tree-shaking")
print("  5. 消息列表无虚拟化，长对话需关注性能")

# 保存数据
snapshot = {
    "timestamp":      datetime.now().isoformat(),
    "notebook":       "12-ui-frontend-metrics",
    "frontend_status": frontend_status,
    "code_stats":     {
        "total_files": code_stats["total_files"],
        "total_loc":   code_stats["total_loc"],
        "component_count": len(code_stats["components"]),
        "page_count": len(code_stats["pages"]),
    },
    "bundle_results": bundle_results,
    "sse_results": sse_results,
    "render_timeline": [
        {"event": t[0], "t_ms": t[1], "desc": t[3]}
        for t in render_timeline
    ],
}

ts = datetime.now().strftime("%Y%m%d_%H%M")
json_path = f"{OUTPUT_DIR}/ui-frontend-metrics_{ts}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(snapshot, f, ensure_ascii=False, indent=2, default=str)

print(f"\n数据已保存: {json_path}")
print("\n12-ui-frontend-metrics.ipynb 执行完成")